# Policy Commons full-census classification audit

Audits every Golden Set record under Protocol 3.6. The input is never modified. Each run creates a new timestamped output directory.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from policycommons_full_census_audit import run

INPUT = Path('/Users/deep1003/Downloads/pcgs_aigov5_20260920/final_release_v6/policycommons_ai_master_final_v6.parquet')
OUTPUT_ROOT = Path('/Users/deep1003/Downloads/pcgs_aigov5_20260920/full_census_audit/runs')
assert INPUT.exists(), INPUT
INPUT

## 1. Run the full-census audit

The vectorised audit scans the complete master and saves its results before plotting.

In [ ]:
metrics = run(INPUT, OUTPUT_ROOT)
run_dir = Path(metrics['output'])
metrics

## 2. Statistical results and regression tests

A failed regression blocks release. Review flags create candidates but do not automatically alter classifications.

In [ ]:
rules = pd.read_csv(run_dir / 'rule_summary.csv')
regressions = pd.read_csv(run_dir / 'regression_results.csv')
candidates = pd.read_csv(run_dir / 'audit_candidates.csv.gz')
display(pd.DataFrame([metrics]))
display(regressions)
display(rules.sort_values('candidate_count', ascending=False).reset_index(drop=True))

In [ ]:
plotted = rules.loc[rules['candidate_count'].gt(0)].sort_values('candidate_count')
ax = plotted.plot.barh(x='rule_id', y='candidate_count', figsize=(11, 8), legend=False, color='#ad791c')
ax.set_title('Full-census audit candidates by rule')
ax.set_xlabel('Records')
ax.set_ylabel('Rule')
plt.tight_layout()
plt.show()

In [ ]:
severity = (rules.groupby('severity', as_index=False)['candidate_count'].sum()
            .sort_values('candidate_count', ascending=False))
display(severity)
ax = severity.plot.bar(x='severity', y='candidate_count', figsize=(8, 4), legend=False, color='#4f6d7a')
ax.set_title('Audit candidates by severity')
ax.set_xlabel('Severity')
ax.set_ylabel('Records')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Candidate rates over time

The annual rate highlights periods where terminology or document form may require a new rule.

In [ ]:
master = pd.read_parquet(INPUT, columns=['record_id', 'year'])
flags = pd.read_parquet(run_dir / 'record_audit_flags.parquet')
yearly = master.merge(flags, on='record_id', validate='one_to_one')
yearly['year_num'] = pd.to_numeric(yearly['year'], errors='coerce')
year_stats = (yearly.dropna(subset=['year_num']).groupby('year_num')
              .agg(records=('record_id', 'size'), flagged=('audit_flag_count', lambda s: int(s.gt(0).sum())))
              .reset_index())
year_stats['flagged_rate'] = year_stats['flagged'] / year_stats['records']
display(year_stats.tail(15))
ax = year_stats.plot(x='year_num', y='flagged_rate', figsize=(11, 4), legend=False, color='#ad791c')
ax.set_title('Audit-candidate rate by publication year')
ax.set_xlabel('Year')
ax.set_ylabel('Flagged share')
plt.tight_layout()
plt.show()

## 4. Review queue and release decision

Inspect high-severity cases first. Confirmed errors and counterexamples must be added to the methodology and regression suite before building another versioned release.

In [ ]:
high_rules = set(rules.loc[rules['severity'].isin(['critical', 'high']), 'rule_id'])
high = candidates[candidates['audit_rule_ids'].fillna('').map(lambda x: bool(set(x.split(';')) & high_rules))]
display(high.head(100))
decision = {'release_blocked': bool((~regressions['passed']).any()), 'failed_regressions': regressions.loc[~regressions['passed'], 'test'].tolist(), 'candidate_records': int(metrics['records_with_flags']), 'high_priority_records': len(high), 'automatic_input_changes': 0, 'run_directory': str(run_dir)}
decision